# Systematic Failure Modes in Small Open-Weight LLMs
## A Controlled Study of Error Taxonomy and In-Context Learning Recovery
### COMP6242 — Deep Learning Group Project

**Abstract:** Large language models fail in consistent, characterisable ways — arithmetic slips,
distractor capture, premise-order sensitivity, hallucinated premises, and logical reversals.
This notebook runs a controlled empirical study coding failure traces into a 7-class taxonomy
and measuring per-error-class recovery across 7 ICL strategies (including a novel
**error-targeted ICL** condition) on open-weight models from Llama-3.2, Qwen2.5, Phi-3.5,
and Gemma-2 at 1–2B, 3B, and 7–9B parameter tiers.

**Primary output:** A reproducible per-error-class × per-ICL-strategy recovery heatmap.

---
**Workflow:**
1. Setup & mount Drive
2. Load datasets (GSM8K family / BBH / FOLIO)
3. Run baseline (S0 zero-shot) + error coding
4. Run ICL strategies S1–S6
5. Compute metrics & generate figures

**Checkpointing:** Results save to Drive every N items. Re-run cells to resume.


## 0. Environment Setup

In [1]:
# Install dependencies (run once per Colab session)
import subprocess, sys

def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

pip_install([
    "transformers>=4.44.0",
    "datasets>=2.20.0",
    "accelerate>=0.30.0",
    "bitsandbytes>=0.43.0",
    "sentencepiece",
    "protobuf",
    "scipy",
    "scikit-learn",
    "pandas",
    "seaborn",
    "tqdm",
    "einops",
    "peft",
])
print("✓ Dependencies installed")


✓ Dependencies installed


## 1. Mount Google Drive & Clone GitHub Repo

In [2]:
from google.colab import drive, userdata
import os, sys

# Mount Drive
drive.mount('/content/drive', force_remount=False)

# ─── EDIT THIS ────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/Adithya-Rama/llm-failure-modes.git"  # <-- your repo
DRIVE_ROOT    = "/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes"
# Safely fetch HF_TOKEN — only needed for gated models (Llama, Gemma).
# Member 1 (Qwen-only) does not need this; userdata.get() raises if the
# secret isn't configured, so we wrap it.
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None
    print('ℹ HF_TOKEN not set in Colab Secrets — fine for no-auth models (qwen-*, phi-3.5). Required for Llama/Gemma.')
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/results", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/figures", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/hf_cache", exist_ok=True)

# Clone / update repo
REPO_DIR = "/content/llm-failure-modes"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {GITHUB_REPO} {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")

# Add src to path
if f"{REPO_DIR}" not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"✓ Drive mounted: {DRIVE_ROOT}")
print(f"✓ Repo at: {REPO_DIR}")


Mounted at /content/drive
✓ Drive mounted: /content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes
✓ Repo at: /content/llm-failure-modes


## 2. Configuration & Imports

In [3]:
import os
os.environ["DRIVE_ROOT"] = DRIVE_ROOT
os.environ["HF_TOKEN"]   = HF_TOKEN or ""
os.environ["TRANSFORMERS_CACHE"] = f"{DRIVE_ROOT}/hf_cache"

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(f"{DRIVE_ROOT}/experiment.log"),
    ]
)

from src.config import (
    MODELS, DATASETS, ERROR_CLASSES, ICL_STRATEGIES, RUN_CONFIG,
    CHECKPOINT_DIR, RESULTS_DIR, FIGURES_DIR, CACHE_DIR
)
from src.data_loader import load_all_datasets, get_dataset_stats
from src.models import load_model, unload_model
from src.inference import run_experiment, run_all_strategies
from src.taxonomy import code_batch, build_error_class_map, sample_for_annotation, export_annotation_csv
from src.metrics import full_metrics_report, save_metrics
from src.visualize import (
    plot_recovery_heatmap, plot_family_comparison,
    plot_scaling_curves, plot_error_distribution,
    plot_robustness_ratios, plot_js_divergence, print_summary_table
)
from src.checkpointing import (
    load_all_checkpoints, print_checkpoint_status,
    export_full_results, get_checkpoint_path
)

print("✓ All imports successful")


✓ All imports successful


## 3. Run Configuration
Edit this cell to control what runs. Start small, scale up.

In [4]:
# -----------------------------------------------------------------------------
# EDIT THIS CELL to control the experiment scope
# Default profile is a tiny smoke test. Switch RUN_PROFILE to "main" only after
# rescoring old checkpoints and confirming parser sanity.
# -----------------------------------------------------------------------------

RUN_PROFILE = "main"  # "smoke" or "main"

# FULL_SCOPE_REFERENCE_DO_NOT_RUN_BY_DEFAULT
# Original ambitious grid retained for documentation/reporting only:
# FULL_SCOPE_MODELS = [
#     "qwen-3b", "llama-3b", "phi-3.5",
#     "qwen-7b", "llama-8b", "gemma-9b",
#     "qwen-1.5b", "llama-1b", "gemma-2b",
# ]
# FULL_SCOPE_DATASETS = [
#     "gsm8k", "gsm_symbolic", "gsm_plus", "gsm_ic",
#     "bbh_logical_deduction", "bbh_tracking", "folio",
# ]
# FULL_SCOPE_STRATEGIES = ["S0", "S1", "S2", "S3", "S4", "S5", "S6"]

# Team split: 3 members × 3 models each = 9 models total
# ← SET THIS TO YOUR MEMBER NUMBER: 1, 2, or 3
RUN_MEMBER = 1

MEMBER_MODELS = {
    # Qwen family — public weights, no HF token needed
    # Gives a clean within-family scaling curve (1.5B → 3B → 7B)
    1: ["qwen-1.5b",  "qwen-3b",   "qwen-7b"],
    # Llama family — gated, needs HF_TOKEN in Colab Secrets
    # Dominant open-source family; cross-family comparison at all 3 tiers
    2: ["llama-1b",   "llama-3b",  "llama-8b"],
    # Phi-3.5 + Gemma — mixed families, mostly gated
    # Phi-3.5 is math-optimised (tests if domain tuning reduces E1)
    3: ["gemma-2b",   "phi-3.5",   "gemma-9b"],
}

FULL_DATASETS = [
    "gsm8k",                 # easy  | clean arithmetic        | E1 baseline
    "gsm_ic",                # easy  | distractor context      | E2 target
    "gsm_symbolic",          # easy  | GSM perturbation        | robustness pair
    "gsm_plus",              # easy  | harder GSM variants     | extra robustness
    "bbh_logical_deduction", # medium| logical deduction       | E3/E7
    "bbh_tracking",          # medium| object tracking         | E3/E4
    "folio",                 # hard  | formal first-order logic| E5/E7
]

# S0→S1→S2→S3→S4→S5(novel)→S6(self-consistency, 5x cost)
FULL_STRATEGIES = ["S0", "S1", "S2", "S3", "S4", "S5", "S6"]

RUN_PHASE2_MODELS = MEMBER_MODELS[1] + MEMBER_MODELS[2] + MEMBER_MODELS[3]

OPTIONAL_ABLATION_MODEL = "qwen-3b"
OPTIONAL_ABLATION_DATASETS = ["gsm8k", "gsm_ic", "folio"]
OPTIONAL_ABLATION_STRATEGIES = ["S5_RANDOM", "S5_CORRECT_ONLY"]  # S6 is in main grid
OPTIONAL_ABLATION_SAMPLES = 50

if RUN_PROFILE == "smoke":
    ACTIVE_MODELS = ["qwen-3b"]
    ACTIVE_DATASETS = ["gsm8k", "gsm_ic", "folio"]
    ACTIVE_STRATEGIES = ["S0", "S1", "S5"]
    N_SAMPLES = 10
elif RUN_PROFILE == "main":
    ACTIVE_MODELS = MEMBER_MODELS[RUN_MEMBER]
    ACTIVE_DATASETS = FULL_DATASETS
    ACTIVE_STRATEGIES = FULL_STRATEGIES
    N_SAMPLES = 100
else:
    raise ValueError("RUN_PROFILE must be 'smoke' or 'main'")

# Checkpoint frequency
CHECKPOINT_EVERY = 50

# Random seed
SEED = 42

# Use 4-bit quantisation (recommended for 7-9B on Colab)
USE_QUANT = True

# ── Per-session single-model control ──────────────────────────────────────
# Each Colab session should run EXACTLY ONE model at a time.
# Set CURRENT_MODEL to a model key (e.g. 'qwen-3b') to run only that model.
# Set to None to run all ACTIVE_MODELS sequentially (only for smoke tests).
#
# Recommended workflow per member:
#   Session 1: CURRENT_MODEL = '<smallest model>'   → ~5-10 hrs
#   Session 2: CURRENT_MODEL = '<3B model>'         → ~10-15 hrs
#   Session 3: CURRENT_MODEL = '<largest model>'    → ~15-25 hrs
#
# Checkpointing means sessions can be interrupted and resumed any time.
CURRENT_MODEL = None   # ← SET THIS: e.g. 'qwen-3b' | None = run all active

if CURRENT_MODEL is not None:
    assert CURRENT_MODEL in ACTIVE_MODELS, \
        f"CURRENT_MODEL '{CURRENT_MODEL}' not in ACTIVE_MODELS: {ACTIVE_MODELS}"
    SESSION_MODELS = [CURRENT_MODEL]
else:
    SESSION_MODELS = ACTIVE_MODELS  # run all (fine for smoke, risky for main)

print(f"{'='*60}")
print(f"  RUN_PROFILE  : {RUN_PROFILE}")
print(f"  RUN_MEMBER   : {RUN_MEMBER}")
print(f"  Session model: {SESSION_MODELS}")
print(f"  Datasets     : {ACTIVE_DATASETS}")
print(f"  Strategies   : {ACTIVE_STRATEGIES}")
print(f"  N samples    : {N_SAMPLES}/dataset")
print(f"  Combos (this session): {len(SESSION_MODELS)} × {len(ACTIVE_DATASETS)} × {len(ACTIVE_STRATEGIES)} = {len(SESSION_MODELS)*len(ACTIVE_DATASETS)*len(ACTIVE_STRATEGIES)}")
print(f"{'='*60}")


Run profile:       main
Active models:     ['qwen-3b', 'phi-3.5', 'qwen-7b']
Active datasets:   ['gsm8k', 'gsm_symbolic', 'gsm_ic', 'folio']
Active strategies: ['S0', 'S1', 'S3', 'S5']
N samples:         100
Optional ablation: qwen-3b ['gsm8k', 'gsm_ic'] ['S5_RANDOM', 'S5_CORRECT_ONLY', 'S6']


## 4. Load Datasets

In [5]:
from src.data_loader import load_dataset_by_key

datasets = {}
failed_datasets = []

for dk in ACTIVE_DATASETS:
    try:
        records = load_dataset_by_key(
            dk,
            n_samples=N_SAMPLES,
            seed=SEED,
            cache_dir=CACHE_DIR,
        )
        if records:
            datasets[dk] = records
            print(f"  ✓ {dk}: {len(records)} items")
        else:
            print(f"  ✗ {dk}: 0 items returned")
            failed_datasets.append(dk)
    except Exception as e:
        print(f"  ✗ {dk}: FAILED ({e})")
        failed_datasets.append(dk)

# Use only successfully loaded datasets
ACTIVE_DATASETS_LOADED = [dk for dk in ACTIVE_DATASETS if dk in datasets]
print(f"\n✓ Loaded {len(datasets)}/{len(ACTIVE_DATASETS)} datasets")
if failed_datasets:
    print(f"  Failed: {failed_datasets}")


README.md: 0.00B [00:00, ?B/s]

  ✓ gsm8k: 100 items


README.md: 0.00B [00:00, ?B/s]

  ✓ gsm_symbolic: 100 items
  ✓ gsm_ic: 100 items


README.md:   0%|          | 0.00/869 [00:00<?, ?B/s]

  ✓ folio: 100 items

✓ Loaded 4/4 datasets


In [6]:
# Preview samples from each dataset
for dk, records in datasets.items():
    cfg = DATASETS[dk]
    print(f"\n{'='*60}")
    print(f"Dataset: {dk} | Tier: {cfg['tier']} | N: {len(records)}")
    print(f"  Q: {records[0]['question'][:150]}...")
    print(f"  A: {records[0]['gold_answer']}")



Dataset: gsm8k | Tier: easy | N: 100
  Q: Darrell and Allen's ages are in the ratio of 7:11. If their total age now is 162, calculate Allen's age 10 years from now....
  A: 109

Dataset: gsm_symbolic | Tier: easy | N: 100
  Q: Sanjay has $72.2 and wants to buy 30 screws from a bin at the home improvement store. Each screw costs $0.94. How much money does Sanjay have left aft...
  A: 44

Dataset: gsm_ic | Tier: easy | N: 100
  Q: Maddy was given 40 chocolate eggs for Easter. She likes to eat two each day after school. If Maddy has two chocolate eggs after school each day, how m...
  A: 4

Dataset: folio | Tier: hard | N: 100
  Q: Premises:
All hydrocarbons are organic compounds .
All alkanes are hydrocarbons
All organic compounds are chemical compounds.
All organic compounds co...
  A: False


## 4.5 Sanity Check — Validate Parser & Prompts Before Running

**Always run this before starting a new model run.**
Generates 3 items through the model for each key strategy and prints:
- The exact prompt sent to the model
- The raw model output
- The extracted answer vs. the gold answer

If `pred_answer` is `None` or wrong for obvious items, stop and debug here before burning Colab credits.

In [ ]:
def sanity_check(model_key: str, n_items: int = 3):
    """
    Quick smoke test: load model, run n_items through S0, S1, S3 on gsm8k,
    print raw output + extracted answer for each. Does NOT save checkpoints.
    Burns ~5-10 min of GPU time but catches parser/prompt/model-load bugs early.
    """
    from src.models import load_model as _lm, format_prompt, generate_response, unload_model as _ul
    from src.prompts import build_prompt
    from src.data_loader import extract_model_answer
    from src.config import DATASETS as DCFG

    test_dk = "gsm8k" if "gsm8k" in datasets else list(datasets.keys())[0]
    test_recs = datasets[test_dk][:n_items]
    answer_type = DCFG[test_dk]["answer_type"]

    print(f"\n{'='*60}")
    print(f"SANITY CHECK | model={model_key} | dataset={test_dk} | n={n_items}")
    print(f"{'='*60}")

    model, tokenizer, model_cfg = _lm(
        model_key, use_quantisation=USE_QUANT,
        hf_token=HF_TOKEN, cache_dir=CACHE_DIR,
    )
    family = model_cfg["family"]

    for strat in ["S0", "S1", "S3"]:
        if strat not in ACTIVE_STRATEGIES:
            continue
        print(f"\n── Strategy: {strat} ──")
        for rec in test_recs:
            msgs = build_prompt(rec, strat, answer_type, seed=SEED)
            prompt_text = format_prompt(msgs, tokenizer, family)
            max_tok = 512 if strat != "S0" else 256
            raw = generate_response(model, tokenizer, prompt_text,
                                    max_new_tokens=max_tok, do_sample=False)[0]
            pred = extract_model_answer(raw, answer_type)
            ok = "✓" if str(pred) == str(rec["gold_answer"]) else "✗"
            print(f"  {ok} gold={rec['gold_answer']} | pred={pred}")
            print(f"     raw (last 200 chars): ...{raw[-200:]!r}")

    _ul(model, tokenizer)
    print("\n✓ Sanity check complete — review pred vs gold above.")
    print("  If pred=None for most items: parser is broken (check raw output format).")
    print("  If pred≠gold for easy items: model load failed, or prompt is wrong.")

# Run on the first session model before main loop. Comment out after confirming.
if SESSION_MODELS:
    sanity_check(SESSION_MODELS[0])


## 5. Check Existing Checkpoints
This shows what's already been computed. Runs resume automatically.

In [7]:
print_checkpoint_status(
    CHECKPOINT_DIR,
    ACTIVE_MODELS,
    ACTIVE_STRATEGIES,
    ACTIVE_DATASETS_LOADED,
)


Model                Strategy gsm8k           gsm_symbolic    gsm_ic          folio          
---------------------------------------------------------------------------------------------
qwen-3b              S0       250             150             150             10             
qwen-3b              S1       10              0               10              10             
qwen-3b              S3       0               0               0               0              
qwen-3b              S5       10              0               10              10             
phi-3.5              S0       0               0               0               0              
phi-3.5              S1       0               0               0               0              
phi-3.5              S3       0               0               0               0              
phi-3.5              S5       0               0               0               0              
qwen-7b              S0       0               0             

## 5.1 Rescore Existing Checkpoints

Run this after parser changes so expensive generations already saved in Drive are reused instead of rerun.


In [8]:
from src.checkpointing import rescore_all_checkpoints

# Rescore all checkpoints for the active model/dataset set.
# This fixes old runs where intermediate equation values were mistaken for final answers.
rescore_summaries = rescore_all_checkpoints(
    CHECKPOINT_DIR,
    model_keys=ACTIVE_MODELS,
    strategy_keys=None,  # include any already-run strategy for these models/datasets
    dataset_keys=ACTIVE_DATASETS_LOADED,
    recode_errors=True,
    save=True,
)

if not rescore_summaries:
    print("No matching checkpoints found to rescore.")
else:
    for s in rescore_summaries:
        print(
            f"{s['path']}: n={s['n']} changed={s['changed']} "
            f"acc {s['before_accuracy']:.3f} -> {s['after_accuracy']:.3f}"
        )


/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S0__folio.json: n=10 changed=0 acc 0.500 -> 0.500
/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S0__gsm8k.json: n=250 changed=0 acc 0.848 -> 0.848
/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S0__gsm_ic.json: n=150 changed=0 acc 0.793 -> 0.793
/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S0__gsm_symbolic.json: n=150 changed=138 acc 0.073 -> 0.753
/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S1__folio.json: n=10 changed=0 acc 0.200 -> 0.200
/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S1__gsm8k.json: n=10 changed=0 acc 0.700 -> 0.700
/content/drive/MyDrive/COMP6242 - Deep Learning/Project/llm-failure-modes/checkpoints/qwen-3b__S1__gsm_ic.json: n=10 

## 6. Run Experiments

**Strategy:**
1. Run S0 (zero-shot baseline) first on ALL models — this provides error classes for S5.
2. Code errors from S0 results.
3. Run S1–S6 (including S5 with error-class-targeted exemplars).

> **Tip:** Run one model at a time. Each model takes ~30–90 min on Colab A100.
> Checkpoints save every N items, so disconnects are safe.


In [9]:
# ─────────────────────────────────────────────────────────────────
# PHASE 1: Baseline (S0 zero-shot) — run on all models first
# ─────────────────────────────────────────────────────────────────
from src.taxonomy import code_batch, build_error_class_map
from src.config import DATASETS as DATASET_CFGS

baseline_error_maps = {}   # {model_key → {dataset_key → {id → error_class}}}

for model_key in SESSION_MODELS:
    print(f"\n{'='*60}")
    print(f"MODEL: {model_key} | STRATEGY: S0 (baseline)")
    print(f"{'='*60}")

    # Check whether each S0 checkpoint covers the currently loaded records.
    # File existence alone is not enough: smoke checkpoints may contain only
    # 10 items while the main profile expects 100.
    from src.checkpointing import load_checkpoint
    all_done = all(
        {r["id"] for r in datasets[dk]}.issubset(
            {r["id"] for r in load_checkpoint(
                get_checkpoint_path(CHECKPOINT_DIR, model_key, "S0", dk)
            )[0]}
        )
        for dk in ACTIVE_DATASETS_LOADED
    )
    if all_done:
        print(f"  ✓ S0 checkpoints already cover active records, skipping inference")
    else:
        model, tokenizer, model_cfg = load_model(
            model_key,
            use_quantisation=USE_QUANT,
            hf_token=HF_TOKEN,
            cache_dir=CACHE_DIR,
        )

        run_all_strategies(
            model, tokenizer, model_cfg, model_key,
            datasets=datasets,
            strategy_keys=["S0"],
            checkpoint_dir=CHECKPOINT_DIR,
            checkpoint_every=CHECKPOINT_EVERY,
            seed=SEED,
        )
        unload_model(model, tokenizer)
        print(f"  ✓ S0 inference complete")

    # Load S0 results and code errors
    from src.checkpointing import load_checkpoint
    error_map_this_model = {}
    for dk in ACTIVE_DATASETS_LOADED:
        ckpt_path = get_checkpoint_path(CHECKPOINT_DIR, model_key, "S0", dk)
        results, _ = load_checkpoint(ckpt_path)
        answer_type = DATASET_CFGS[dk]["answer_type"]
        results = code_batch(results, answer_type)
        # Re-save with error codes
        from src.checkpointing import save_checkpoint
        save_checkpoint(ckpt_path, results)
        error_map_this_model[dk] = build_error_class_map(results)
        n_errors = sum(1 for r in results if not r["correct"])
        print(f"    {dk}: {len(results)} items, {n_errors} errors coded")

    baseline_error_maps[model_key] = error_map_this_model

print("\n✓ Phase 1 (S0 baseline + error coding) complete")



MODEL: qwen-3b | STRATEGY: S0 (baseline)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2.5-3B|S0|folio:   0%|          | 0/90 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Qwen2.5-3B|S0|folio: 100%|██████████| 90/90 [20:21<00:00, 13.57s/it, acc=0.517]


  ✓ S0 inference complete
    gsm8k: 250 items, 38 errors coded
    gsm_symbolic: 150 items, 37 errors coded
    gsm_ic: 150 items, 31 errors coded
    folio: 100 items, 51 errors coded

MODEL: phi-3.5 | STRATEGY: S0 (baseline)


...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2.5-7B|S0|gsm8k: 100%|██████████| 100/100 [23:15<00:00, 13.96s/it, acc=0.560]
Qwen2.5-7B|S0|gsm_symbolic: 100%|██████████| 100/100 [23:08<00:00, 13.89s/it, acc=0.480]
Qwen2.5-7B|S0|gsm_ic: 100%|██████████| 100/100 [23:30<00:00, 14.11s/it, acc=0.570]
Qwen2.5-7B|S0|folio: 100%|██████████| 100/100 [03:54<00:00,  2.35s/it, acc=0.540]


  ✓ S0 inference complete
    gsm8k: 100 items, 44 errors coded
    gsm_symbolic: 100 items, 52 errors coded
    gsm_ic: 100 items, 43 errors coded
    folio: 100 items, 46 errors coded

✓ Phase 1 (S0 baseline + error coding) complete


In [10]:
# ─────────────────────────────────────────────────────────────────
# PHASE 2: Active ICL strategies
# ─────────────────────────────────────────────────────────────────
ICL_ONLY = [s for s in ACTIVE_STRATEGIES if s != "S0"]
PHASE2_MODELS = [m for m in SESSION_MODELS if m in RUN_PHASE2_MODELS]

for model_key in PHASE2_MODELS:
    print(f"\n{'='*60}")
    print(f"MODEL: {model_key} | STRATEGIES: {ICL_ONLY}")
    print(f"{'='*60}")

    # Build S5 error class map for this model
    s5_error_classes = baseline_error_maps.get(model_key, {})

    model, tokenizer, model_cfg = load_model(
        model_key,
        use_quantisation=USE_QUANT,
        hf_token=HF_TOKEN,
        cache_dir=CACHE_DIR,
    )

    run_all_strategies(
        model, tokenizer, model_cfg, model_key,
        datasets=datasets,
        strategy_keys=ICL_ONLY,
        checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_every=CHECKPOINT_EVERY,
        baseline_errors=s5_error_classes,  # used by S5
        seed=SEED,
    )

    unload_model(model, tokenizer)
    print(f"  ✓ {model_key} complete")

print("\n✓ Phase 2 (active ICL strategies) complete")



MODEL: qwen-3b | STRATEGIES: ['S1', 'S3', 'S5']


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Qwen2.5-3B|S1|gsm8k: 100%|██████████| 90/90 [24:04<00:00, 16.05s/it, acc=0.650]
Qwen2.5-3B|S1|gsm_symbolic: 100%|██████████| 100/100 [27:44<00:00, 16.65s/it, acc=0.520]
Qwen2.5-3B|S1|gsm_ic: 100%|██████████| 90/90 [24:30<00:00, 16.33s/it, acc=0.633]
Qwen2.5-3B|S1|folio: 100%|██████████| 90/90 [27:54<00:00, 18.61s/it, acc=0.333]
Qwen2.5-3B|S3|gsm8k: 100%|██████████| 100/100 [19:41<00:00, 11.82s/it, acc=0.770]
Qwen2.5-3B|S3|gsm_symbolic: 100%|██████████| 100/100 [19:07<00:00, 11.47s/it, acc=0.720]
Qwen2.5-3B|S3|gsm_ic: 100%|██████████| 100/100 [18:59<00:00, 11.40s/it, acc=0.790]
Qwen2.5-3B|S3|folio: 100%|██████████| 100/100 [23:16<00:00, 13.96s/it, acc=0.360]
Qwen2.5-3B|S5|gsm8k: 100%|██████████| 90/90 [17:46<00:00, 11.85s/it, acc=0.717]
Qwen2.5-3B|S5|gsm_symbolic: 100%|██████████| 100/100 [19:21<00:00, 11.61s/it, acc=0.710]
Qwen2.5-3B|S5|gsm_ic: 100%|██████████| 90/90 [17:56<00:00, 11.96s/it, acc=0.783]
Qwen2.5-3B|S5|folio: 100%|██████████| 90/90 [22:09<00:00, 14.78s/it, acc=0.317]


  ✓ qwen-3b complete

MODEL: phi-3.5 | STRATEGIES: ['S1', 'S3', 'S5']


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

  ✓ phi-3.5 complete

MODEL: qwen-7b | STRATEGIES: ['S1', 'S3', 'S5']


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2.5-7B|S1|gsm8k: 100%|██████████| 100/100 [23:20<00:00, 14.01s/it, acc=0.570]
Qwen2.5-7B|S1|gsm_symbolic: 100%|██████████| 100/100 [23:43<00:00, 14.23s/it, acc=0.500]
Qwen2.5-7B|S1|gsm_ic:   0%|          | 0/100 [00:08<?, ?it/s]


KeyboardInterrupt: 

## 7. Inter-Annotator Agreement (Cohen's κ)
Export a sample CSV for human labelling, then compute κ after both annotators label it.

In [ ]:
from src.checkpointing import load_checkpoint
from src.taxonomy import sample_for_annotation, export_annotation_csv, compute_kappa
from src.config import DATASETS as DATASET_CFGS

# Collect a sample of S0 failures across all models for annotation
all_s0_failures = []
for model_key in SESSION_MODELS:  # this member's models only
    for dk in ACTIVE_DATASETS_LOADED:
        ckpt = get_checkpoint_path(CHECKPOINT_DIR, model_key, "S0", dk)
        results, _ = load_checkpoint(ckpt)
        all_s0_failures.extend([r for r in results if not r.get("correct")])

sample = sample_for_annotation(all_s0_failures, n=150, seed=SEED)
annotation_csv = f"{RESULTS_DIR}/annotation_sample.csv"
export_annotation_csv(sample, annotation_csv)
print(f"✓ Annotation CSV exported to: {annotation_csv}")
print(f"  {len(sample)} items for human labelling")
print("  → Fill in the 'human_label' column and re-upload to Drive")

# ── After human labelling, load and compute kappa ──
# Uncomment and run this block after annotation is complete:
#
# import pandas as pd
# ann_df = pd.read_csv(annotation_csv)
# auto_labels = ann_df["auto_label"].tolist()
# human_labels = ann_df["human_label"].tolist()
# # Drop rows where human didn't annotate
# paired = [(a, h) for a, h in zip(auto_labels, human_labels) if pd.notna(h) and h.strip()]
# if paired:
#     auto, human = zip(*paired)
#     kappa = compute_kappa(list(auto), list(human))
#     print(f"\nCohen's κ = {kappa:.3f}")
#     if kappa >= 0.8: print("  → Excellent agreement")
#     elif kappa >= 0.6: print("  → Substantial agreement")
#     else: print("  → Moderate agreement — review taxonomy")


## 8. Compute All Metrics

In [ ]:
from src.checkpointing import load_all_checkpoints, filter_all_results_to_records
from src.metrics import full_metrics_report, save_metrics

# ── Aggregation scope ────────────────────────────────────────────────────
# To produce the FINAL paper figures we want metrics across ALL 9 models —
# not just this member's 3.  Read whatever checkpoints exist in Drive.
ALL_MODELS_FOR_REPORT = MEMBER_MODELS[1] + MEMBER_MODELS[2] + MEMBER_MODELS[3]

print("Loading all results from checkpoints...")
all_results = load_all_checkpoints(CHECKPOINT_DIR)
all_results = filter_all_results_to_records(all_results, datasets)

# Quick status: which models actually have data?
present_models = sorted(all_results.keys())
missing_models = [m for m in ALL_MODELS_FOR_REPORT if m not in present_models]
total = sum(len(v) for m in all_results.values() for s in m.values() for v in s.values())
print(f"✓ {total} total results loaded")
print(f"✓ Models present in checkpoints: {present_models}")
if missing_models:
    print(f"⚠ Missing models (not yet run by their member): {missing_models}")

# Compute metrics across ALL 9 models (missing ones contribute NaN, no error)
metrics = full_metrics_report(
    all_results,
    model_keys=ALL_MODELS_FOR_REPORT,
    strategy_keys=ACTIVE_STRATEGIES,
    dataset_keys=ACTIVE_DATASETS_LOADED,
    model_configs=MODELS,
)

save_metrics(metrics, RESULTS_DIR)
print("✓ Metrics computed and saved")


In [ ]:
# Print summary tables
from src.visualize import print_summary_table
print_summary_table(metrics["accuracy"], ACTIVE_STRATEGIES)

# Show recovery delta
print("\n=== Recovery Delta (per error class × strategy) ===")
print(metrics["recovery"].round(3).to_string())


## 9. Generate All Figures

In [ ]:
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

# Fig 1: Main recovery heatmap (the paper's key figure)
print("Generating Fig 1: Recovery Heatmap...")
plot_recovery_heatmap(metrics["recovery"], figures_dir=FIGURES_DIR)

# Fig 2: Family comparison at 3B tier
print("Generating Fig 2: Family Comparison...")
plot_family_comparison(
    metrics["accuracy"],
    size_tier="3B",
    strategy_keys=["S0", "S1", "S3", "S5", "S6"],
    figures_dir=FIGURES_DIR,
)

# Fig 3: Scaling curves
print("Generating Fig 3: Scaling Curves...")
plot_scaling_curves(metrics["accuracy"], figures_dir=FIGURES_DIR)

# Fig 4: Error distribution at baseline
print("Generating Fig 4: Error Distribution...")
plot_error_distribution(metrics["error_dist"], strategy="S0", figures_dir=FIGURES_DIR)

# Fig 5: Robustness ratios
print("Generating Fig 5: Robustness Ratios...")
plot_robustness_ratios(metrics["robustness"], figures_dir=FIGURES_DIR)

# Fig 6: JS divergence
print("Generating Fig 6: JS Divergence...")
plot_js_divergence(
    all_results, ALL_MODELS_FOR_REPORT, ACTIVE_STRATEGIES,
    ACTIVE_DATASETS_LOADED, MODELS, figures_dir=FIGURES_DIR,
)

print(f"\n✓ All figures saved to {FIGURES_DIR}")


## 10. Export Full Results & Archive

In [ ]:
from src.checkpointing import export_full_results
import json, time

timestamp = time.strftime("%Y%m%d_%H%M")
archive_path = f"{RESULTS_DIR}/all_results_{timestamp}.json"
export_full_results(all_results, archive_path)
print(f"✓ Full results archived to {archive_path}")

# Export metrics as CSV
metrics["accuracy"].to_csv(f"{RESULTS_DIR}/accuracy_{timestamp}.csv")
metrics["recovery"].to_csv(f"{RESULTS_DIR}/recovery_heatmap_{timestamp}.csv")
print("✓ CSVs exported")

# Summary
print(f"\n{'='*60}")
print("EXPERIMENT COMPLETE")
print(f"{'='*60}")
print(f"Figures dir:  {FIGURES_DIR}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Checkpoints:  {CHECKPOINT_DIR}")
print(f"Archive:      {archive_path}")


## 11. Optional Ablations: Targeting and Self-Consistency

Run after the main qwen-3b S0 baseline exists. This keeps the HD novelty claims honest without expanding the full grid.


In [ ]:
# Optional ablations: S5 random target, S5 correct-only, and S6 on a 50-item subset
from src.inference import run_all_strategies
from src.taxonomy import build_error_class_map, code_batch
from src.checkpointing import load_checkpoint
from src.config import DATASETS as DATASET_CFGS

ABLATION_MODEL = OPTIONAL_ABLATION_MODEL
ABLATION_DATASETS = [dk for dk in OPTIONAL_ABLATION_DATASETS if dk in datasets]
ABLATION_STRATEGIES = OPTIONAL_ABLATION_STRATEGIES
ABLATION_N = OPTIONAL_ABLATION_SAMPLES

ablation_datasets = {
    dk: datasets[dk][:min(ABLATION_N, len(datasets[dk]))]
    for dk in ABLATION_DATASETS
}

# Build baseline error maps from existing S0 checkpoints.
ablation_error_maps = {}
for dk in ABLATION_DATASETS:
    ckpt = get_checkpoint_path(CHECKPOINT_DIR, ABLATION_MODEL, "S0", dk)
    results, _ = load_checkpoint(ckpt)
    if not results:
        raise RuntimeError(f"Missing S0 checkpoint for {ABLATION_MODEL}/{dk}; run Phase 1 first.")
    answer_type = DATASET_CFGS[dk]["answer_type"]
    code_batch(results, answer_type)
    ablation_error_maps[dk] = build_error_class_map(results)

model, tokenizer, model_cfg = load_model(
    ABLATION_MODEL,
    use_quantisation=USE_QUANT,
    hf_token=HF_TOKEN,
    cache_dir=CACHE_DIR,
)

run_all_strategies(
    model, tokenizer, model_cfg, ABLATION_MODEL,
    datasets=ablation_datasets,
    strategy_keys=ABLATION_STRATEGIES,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=CHECKPOINT_EVERY,
    baseline_errors=ablation_error_maps,
    seed=SEED,
)

unload_model(model, tokenizer)
print("✓ Optional ablations complete")


## 12. Statistical Significance (McNemar's Test)

In [ ]:
from src.metrics import mcnemar_test
from src.checkpointing import load_checkpoint

# Compare S0 vs S5 (novel strategy) for statistical significance
# Run on first model, first dataset with enough items

MODEL_FOR_TEST = "qwen-3b"
DATASET_FOR_TEST = "gsm8k"

s0_ckpt = get_checkpoint_path(CHECKPOINT_DIR, MODEL_FOR_TEST, "S0", DATASET_FOR_TEST)
s5_ckpt = get_checkpoint_path(CHECKPOINT_DIR, MODEL_FOR_TEST, "S5", DATASET_FOR_TEST)

s0_results, _ = load_checkpoint(s0_ckpt)
s5_results, _ = load_checkpoint(s5_ckpt)

if s0_results and s5_results:
    # Align by record ID
    s0_by_id = {r["id"]: r for r in s0_results}
    s5_by_id = {r["id"]: r for r in s5_results}
    common_ids = sorted(set(s0_by_id) & set(s5_by_id))

    s0_aligned = [s0_by_id[i] for i in common_ids]
    s5_aligned = [s5_by_id[i] for i in common_ids]

    chi2, p = mcnemar_test(s0_aligned, s5_aligned)
    print(f"McNemar's Test: S0 vs S5 on {MODEL_FOR_TEST} / {DATASET_FOR_TEST}")
    print(f"  χ² = {chi2:.3f}, p = {p:.4f}")
    print(f"  {'Significant (p < 0.05)' if p < 0.05 else 'Not significant (p >= 0.05)'}")
else:
    print("Checkpoints not found — run experiments first.")
